In [1]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

In [2]:
# !pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.6 MB/s eta 0:00:00


In [3]:
from langchain_groq import ChatGroq

In [4]:
model = ChatGroq(model="qwen/qwen3.8-27b", api_key=GROQ_API_KEY)

In [5]:
from langchain_core.prompts import PromptTemplate

### 1. String Output Parser

In [6]:
template1 = PromptTemplate(
    template='Write a detailed report on {topic}',
    input_variables=['topic']
)

In [7]:
template2 = PromptTemplate(
    template='Write a 5 line summary of following text. /n {text}',
    input_variables=['text']
)

In [8]:
from langchain_core.output_parsers import StrOutputParser

In [9]:
strparser = StrOutputParser()

In [10]:
chain = template1 | model | strparser | template2 | model | strparser

In [11]:
result = chain.invoke({'topic': 'Generative AI'})

In [ ]:
print(result)

Generative AI marks a paradigm shift by enabling systems to create original content across various media, transforming it from a research concept into a major economic force.
The technology relies on advanced architectures like Large Language Models and Diffusion Models, trained through massive data pre-training and human feedback alignment.
It drives significant productivity gains and innovation in industries ranging from healthcare and software development to creative arts and education.
However, its rapid adoption raises critical ethical challenges regarding intellectual property, misinformation, bias, environmental costs, and labor displacement.
Moving forward, the field is heading toward multimodal integration and autonomous agents, necessitating robust regulatory frameworks and strategic governance for responsible use.


### 2. JSON Output Parser

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
jsonparser = JsonOutputParser()

In [ ]:
template3 = PromptTemplate(
    template = 'Give the company name, creator name and founded year of any Gen AI model \n {format_instruction}',
    input_variables=[],
    partial_variables={'format_instruction': jsonparser.get_format_instructions()}
)

In [ ]:
json_chain = template3 | model | jsonparser
result = json_chain.invoke({})

In [ ]:
print(result)

{'company_name': 'OpenAI', 'creator_name': 'Sam Altman', 'founded_year': 2015}


### 3. Structured Output Parser

In [ ]:
# !pip install langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.0 MB/s eta 0:00:00


In [ ]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema

In [ ]:
schema = [
    ResponseSchema(name='Model_name', description='The name of the model'),
    ResponseSchema(name='Model_version', description='The version of the model'),
    ResponseSchema(name='Creator_name', description='The name of the creator'),
    ResponseSchema(name='Company_name', description='The name of the company'),
    ResponseSchema(name='Founded_year', description='The year the company was founded')
]

In [ ]:
structured_parser = StructuredOutputParser.from_response_schemas(schema)

In [ ]:
template4 = PromptTemplate(
    template = 'Give the company name, creator name, latest model name, model version and founded year of any Gen AI model \n {format_instruction}',
    input_variables=[],
    partial_variables={'format_instruction': structured_parser.get_format_instructions()}
)

In [ ]:
structured_chain = template4 | model  | structured_parser

result = structured_chain.invoke({})
print(result)

{'Model_name': 'GPT-4', 'Model_version': '4', 'Creator_name': 'OpenAI', 'Company_name': 'OpenAI', 'Founded_year': '2015'}


### 4. Pydantic Output Parser

> PydanticOutputParser is a structured output parser in LangChain that uses Pydantic models to enforce schema validation when processing LLM responses.



In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

In [ ]:
class Company(BaseModel):
    company_name: str = Field(description='The name of the company')
    creator_name: str = Field(description='The name of the creator')
    model_name: str = Field(description='The name of the model')
    founded_year: int = Field(description='The year the company was founded')
    model_version: str = Field(description='The version of the model')
    open_source: bool = Field(description='Whether the model is open source')

In [ ]:
pydantic_parser = PydanticOutputParser(pydantic_object=Company)

In [ ]:
template5 = PromptTemplate(
    template = 'Give the company name, creator name, latest model name, model version and founded year of any Gen AI model. and also tell whether this is open source model or not \n {format_instruction}',
    input_variables=[],
    partial_variables={'format_instruction': pydantic_parser.get_format_instructions()}
)

In [ ]:
pydantic_chain = template5 | model | pydantic_parser
result = pydantic_chain.invoke({})
print(result)

company_name='Meta' creator_name='Meta AI' model_name='Llama 3' founded_year=2004 model_version='3' open_source=True


In [ ]:
print(type(result))

<class '__main__.Company'>
